# 02 — Temel NLP Görevleri

**Kapsam:** Metin sınıflandırma, varlık tanıma, özetleme, soru-cevap, diyalog yönetimi
ve makine çevirisi gibi NLP görevleri için uçtan uca çözümler.

Bu notebook, `src/nlp_tasks/` altındaki her modülü tek tek çalıştırıp çıktısını inceler.

In [ ]:
# Bu hücre HER notebook'ta ayrı ayrı çalıştırılmalı: Colab'da her sekme/notebook
# genellikle kendi çalışma zamanını (VM) alır, yani /content her seferinde sıfırdanmış
# gibi başlar. Bu hücre kendi kendini onaran bir kurulum yapar:
#   1) Proje klasörü zaten varsa (aynı çalışma zamanında önceki hücre/notebook
#      tarafından kurulmuşsa) hiçbir şey yapmadan devam eder.
#   2) Yoksa Google Drive'ı mount edip, DRIVE_ZIP_PATH'teki zip'i /content'e açar
#      (zip'in içinde 'baykar-nlp-hazirlik/' klasörü kök olarak yer almalı).
#   3) Drive'da zip de yoksa, kendi GitHub reponuzu klonlamanız için bir uyarı basar.
import os, sys

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/baykar-nlp-hazirlik.zip"

if not os.path.exists(PROJECT_DIR):
    try:
        from google.colab import drive
        # drive.mount() zaten mount edilmişse anında geri döner (idempotent);
        # os.path.exists("/content/drive") ile "mount edilmiş mi" kontrol etmek
        # güvenilmez çünkü klasör, başarısız/yarım bir mount denemesinden sonra
        # bile var olabilir. Bu yüzden koşulsuz çağırıyoruz.
        drive.mount("/content/drive", force_remount=True)
        if os.path.exists(DRIVE_ZIP_PATH):
            import shutil
            shutil.unpack_archive(DRIVE_ZIP_PATH, "/content")
        else:
            print(f"UYARI: {DRIVE_ZIP_PATH} bulunamadı. Zip'i Drive'ınızın köküne "
                  "yükleyin ya da kendi reponuzu klonlayın: "
                  f"!git clone <repo-url> {PROJECT_DIR}")
    except ImportError:
        pass  # Colab dışında (yerelde) çalışıyorsanız bu adım gerekmez.

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)


## Sınıflandırma (zero-shot)

Etiketli veri toplamadan önce hızlı bir başlangıç noktası: doküman türünü tahmin eder.

In [ ]:
from src.nlp_tasks.classification import zero_shot_classify

text = "Bu bölümde sistemin ana bileşenleri ve bunlar arasındaki veri akışı açıklanmaktadır."
result = zero_shot_classify(text)
print(result["label"])
print(result["scores"])


## Varlık Tanıma (NER)

Genel model + teknik-dokümantasyona özgü regex desenleri (kod parçası, CLI bayrağı, sürüm, dosya yolu) birleşimi.

In [ ]:
from src.nlp_tasks.ner import extract_entities

sample = "`kullanici_olustur(ad, e_posta)` fonksiyonunu çağırmadan önce DATABASE_URL ortam değişkenini v2.3.0 sürümünde /etc/proje/config.yaml içinde ayarlayın."
for ent in extract_entities(sample):
    print(ent)


## Özetleme

Extractive (hızlı, halüsinasyonsuz) vs. abstractive (akıcı) karşılaştırması.

In [ ]:
from src.nlp_tasks.summarization import summarize

with open("data/raw/corpus.jsonl", encoding="utf-8") as f:
    import json
    long_text = json.loads(f.readline())["text"]

print("EXTRACTIVE:\n", summarize(long_text, method="extractive", sentence_count=3))
print("\nABSTRACTIVE:\n", summarize(long_text, method="abstractive"))


## Soru-Cevap (extractive)

In [ ]:
from src.nlp_tasks.qa import extractive_answer

ctx = "kullanici_olustur fonksiyonu, e_posta parametresi zaten kayıtlıysa DuplicateEmailError fırlatır."
print(extractive_answer("kullanici_olustur hangi hatayı fırlatır?", ctx))


## Makine Çevirisi

In [ ]:
from src.nlp_tasks.translation import translate

print(translate("Bu fonksiyon, kullanıcı e-postası zaten kayıtlıysa bir hata fırlatır.", "tr-en"))


## Diyalog Yönetimi

`dialogue.py`, RAG pipeline'ına (03. notebook'ta kuracağız) bağımlıdır — vektör veritabanı
indekslenmeden bu hücre çalışmaz. Şimdilik sadece bağlam yeniden yazma (query rewriting)
mekanizmasını inceleyelim.

In [ ]:
from src.nlp_tasks.dialogue import DialogueSession, rewrite_standalone_query

session = DialogueSession(session_id="demo")
session.add_user_turn("İyi bir README nasıl yapılandırılır?")
session.add_assistant_turn("README; özellikler, kurulum, kullanım ve lisans bölümlerini içermelidir.")

standalone = rewrite_standalone_query(session, "Peki kurulum bölümüne ne yazmalıyım?")
print(standalone)
